# Fase di prototipazione

Questo notebook è la **fase di prototipazione** del data agent: esplorazione della qualità dei dati, prime regole di pulizia e primi tentativi con il pandas agent di LangChain.

Il servizio usato dal progetto è `data_agent/app.py`, con la pulizia in `pulizia.py`, i calcoli deterministici in `briefing.py` e i grafici in `grafici.py`. Il notebook resta come documentazione del percorso e non è aggiornato: il prompt è stato poi modificato per il servizio FastAPI, quindi rieseguire le celle con `.format()` può dare KeyError.

# Prototipo Data Agent – Esplorazione della qualità dei dati

In questa fase **non** creiamo ancora nessun agente AI: prepariamo l'ambiente e analizziamo la qualità del dataset `dati_negozi.csv`.

## Sezione 1 – Setup

Carichiamo il file `.env` dalla cartella principale del progetto con `python-dotenv` e verifichiamo che la variabile `OPENAI_API_KEY` sia presente.
Per sicurezza la chiave **non viene mai stampata**, nemmeno in parte: mostriamo solo se è stata trovata oppure no.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Il notebook si trova in data_agent/, quindi la cartella principale è quella superiore
CARTELLA_PROGETTO = Path.cwd().parent
load_dotenv(CARTELLA_PROGETTO / ".env")

if os.getenv("OPENAI_API_KEY"):
    print("Chiave trovata")
else:
    print("Chiave mancante")

Chiave trovata


## Sezione 2 – Caricamento

Carichiamo `../data/dati_negozi.csv` con pandas e diamo un primo sguardo: dimensioni, tipi delle colonne e prime righe.
Leggiamo il file senza conversioni automatiche delle date, così da poter controllare i formati originali.

In [2]:
df = pd.read_csv(CARTELLA_PROGETTO / "data" / "dati_negozi.csv")

print(f"Dimensioni: {df.shape[0]} righe x {df.shape[1]} colonne")

Dimensioni: 4375 righe x 18 colonne


In [3]:
df.dtypes

data                        str
negozio                     str
vendite_totali          float64
obiettivo_vendite       float64
numero_scontrini        float64
vendite_ortofrutta      float64
vendite_panetteria      float64
vendite_freschi         float64
scarti_ortofrutta       float64
scarti_panetteria       float64
scarti_freschi          float64
scarti_kg_totali        float64
referenze_in_rottura      int64
consegne_in_ritardo       int64
ore_pianificate         float64
ore_assenza             float64
eccedenze_donate_kg     float64
evento                      str
dtype: object

In [4]:
df.head()

,data,negozio,vendite_totali,obiettivo_vendite,numero_scontrini,vendite_ortofrutta,vendite_panetteria,vendite_freschi,scarti_ortofrutta,scarti_panetteria,scarti_freschi,scarti_kg_totali,referenze_in_rottura,consegne_in_ritardo,ore_pianificate,ore_assenza,eccedenze_donate_kg,evento
0,2024-09-01,NEG01,30539.76,32810.0,1203.0,3468.65,1568.46,8341.45,181.11,82.38,187.84,126.7,36,0,213.2,9.8,21.7,nessuno
1,2024-09-01,NEG02,28211.11,27760.0,1081.0,3006.55,1451.86,7854.30,135.78,102.68,168.66,109.8,46,0,208.8,6.8,14.9,nessuno
2,2024-09-01,NEG03,33321.52,37850.0,1354.0,3806.47,1789.55,9050.12,175.79,84.49,199.37,126.2,40,1,210.9,10.3,27.2,nessuno
3,01/09/2024,NEG04,24228.23,23970.0,976.0,2682.33,1210.58,6249.84,108.87,42.46,136.26,76.8,31,0,211.8,11.8,14.6,promo
4,2024-09-01,NEG05,31108.53,31540.0,1152.0,3447.08,1683.87,8710.85,138.94,185.10,190.22,137.2,49,0,210.9,7.9,28.9,nessuno


## Sezione 3 – Esplorazione della qualità dei dati

Ogni cella esegue un controllo e stampa quante righe sono coinvolte e qualche esempio.
In questa fase **non modifichiamo** i dati: ci limitiamo a osservare i problemi.

In [5]:
NUM_ESEMPI = 5

def mostra_esempi(righe, colonne=None):
    """Stampa il numero di righe coinvolte e alcuni esempi."""
    print(f"Righe coinvolte: {len(righe)}")
    if len(righe) > 0:
        esempi = righe if colonne is None else righe[colonne]
        print(esempi.head(NUM_ESEMPI).to_string())

### a) Valori mancanti per colonna

In [6]:
mancanti = df.isna().sum()
mancanti = mancanti[mancanti > 0].sort_values(ascending=False)
print("Valori mancanti per colonna:")
print(mancanti.to_string() if len(mancanti) else "nessuno")
print()
mostra_esempi(df[df.isna().any(axis=1)])

Valori mancanti per colonna:
scarti_panetteria    44
numero_scontrini     43
scarti_ortofrutta    43
scarti_freschi       43

Righe coinvolte: 171
           data negozio  vendite_totali  obiettivo_vendite  numero_scontrini  vendite_ortofrutta  vendite_panetteria  vendite_freschi  scarti_ortofrutta  scarti_panetteria  scarti_freschi  scarti_kg_totali  referenze_in_rottura  consegne_in_ritardo  ore_pianificate  ore_assenza  eccedenze_donate_kg   evento
36   2024-09-06   NEG06        24746.53            26660.0               NaN             2644.07             1192.15          6445.25             125.99              63.59          139.23              90.9                    39                    0            185.9          5.5                 14.5  nessuno
58   2024-09-10   NEG04        16565.33            17270.0             682.0             1701.80              841.45          4948.44             117.95              48.58             NaN              77.7                    39        

### b) Righe duplicate

Consideriamo duplicate le righe identiche in tutte le colonne (conteggiamo le copie in eccesso, esclusa la prima occorrenza).

In [7]:
duplicate = df[df.duplicated(keep="first")]
mostra_esempi(duplicate, ["data", "negozio", "vendite_totali", "evento"])

Righe coinvolte: 43
           data negozio  vendite_totali   evento
23   2024-09-04   NEG05        22395.62  nessuno
200  2024-10-04   NEG01        27331.94  nessuno
211  2024-10-05   NEG05        33471.89  nessuno
230  2024-10-08   NEG05        22369.62  nessuno
322  2024-10-23   NEG06        21931.99  nessuno


### c) Formati di data diversi da AAAA-MM-GG

In [8]:
formato_ok = df["data"].astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}")
date_non_standard = df[~formato_ok]
mostra_esempi(date_non_standard, ["data", "negozio"])
print()
print("Esempi di formati trovati:", date_non_standard["data"].unique()[:10].tolist())

Righe coinvolte: 129
           data negozio
3    01/09/2024   NEG04
48   08/09/2024   NEG06
88   15-09-2024   NEG04
99   17/09/2024   NEG03
154  26/09/2024   NEG04

Esempi di formati trovati: ['01/09/2024', '08/09/2024', '15-09-2024', '17/09/2024', '26/09/2024', '29/09/2024', '08-10-2024', '18-10-2024', '23-10-2024', '24/10/2024']


### d) Valori della colonna negozio diversi da NEG01...NEG06

In [9]:
NEGOZI_VALIDI = [f"NEG0{i}" for i in range(1, 7)]
negozio_non_valido = df[~df["negozio"].isin(NEGOZI_VALIDI)]
mostra_esempi(negozio_non_valido, ["data", "negozio"])
print()
print("Varianti trovate (repr per vedere eventuali spazi):")
print(negozio_non_valido["negozio"].map(repr).value_counts().to_string())

Righe coinvolte: 132
          data    negozio
11  2024-09-02     neg06 
28  2024-09-05  Negozio 4
29  2024-09-05  Negozio 6
30  2024-09-05     neg_02
72  2024-09-12  Negozio 5

Varianti trovate (repr per vedere eventuali spazi):
negozio
'negozio 3'    15
'NEG 03'       14
'Negozio 6'    13
'neg01'        13
'Negozio 5'    11
'neg04'        11
'NEG-05'       11
'neg06 '       10
'neg_02'       10
'Negozio 4'     9
'Negozio 1'     8
'Negozio 2'     7


### e) vendite_totali negative

In [10]:
vendite_negative = df[df["vendite_totali"] < 0]
mostra_esempi(vendite_negative, ["data", "negozio", "vendite_totali"])

Righe coinvolte: 5
            data negozio  vendite_totali
609   2024-12-10   NEG03       -31801.78
948   2025-02-07   NEG05       -24367.93
1785  2025-06-27   NEG02       -23313.42
3753  2026-05-22   NEG03       -35513.16
4330  2026-08-24   NEG05       -21900.86


### f) vendite_totali anomale (più di 3 volte la mediana del proprio negozio)

Poiché la colonna `negozio` contiene varianti dello stesso codice (es. `neg01`, `Negozio 1`), per calcolare la mediana usiamo un codice normalizzato **temporaneo** (estraiamo la cifra del negozio). Il DataFrame originale non viene modificato.

In [11]:
codice_negozio = "NEG0" + df["negozio"].astype(str).str.extract(r"(\d)\s*$", expand=False)
mediana_negozio = df.groupby(codice_negozio)["vendite_totali"].transform("median")

anomale = df[df["vendite_totali"] > 3 * mediana_negozio].copy()
anomale["mediana_negozio"] = mediana_negozio[anomale.index]
mostra_esempi(anomale, ["data", "negozio", "vendite_totali", "mediana_negozio"])

Righe coinvolte: 5
            data    negozio  vendite_totali  mediana_negozio
6     2024-09-02      NEG01        201372.5         26211.86
1297  2025-04-06      NEG04        244427.7         19283.64
1578  2025-05-23  Negozio 1        323842.5         26211.86
3240  2026-02-25      NEG03        252445.4         30077.55
3783  2026-05-27      NEG01        261952.6         26211.86


### g) ore_assenza maggiori di ore_pianificate

In [12]:
assenze_eccessive = df[df["ore_assenza"] > df["ore_pianificate"]]
mostra_esempi(assenze_eccessive, ["data", "negozio", "ore_pianificate", "ore_assenza"])

Righe coinvolte: 5
            data negozio  ore_pianificate  ore_assenza
261   2024-10-13   NEG06            219.5       307.30
1065  2025-02-27   NEG01            175.5       245.70
1382  2025-04-21   NEG05            167.8       234.92
2633  2025-11-14   NEG02            198.0       277.20
3149  2026-02-10   NEG02            164.8       230.72


## Sezione 4 – Regole di pulizia

Le regole sono implementate in `pulizia.py` (funzione `pulisci_dati`) e vengono applicate **in quest'ordine**, solo in memoria: il CSV originale non viene modificato.

a) **Date**: conversione in datetime gestendo esplicitamente i tre formati presenti (AAAA-MM-GG, GG/MM/AAAA, GG-MM-AAAA), senza riconoscimento automatico.

b) **Negozi**: nomi uniformati a NEG01…NEG06 ricavando il numero dal testo (ignorando spazi, maiuscole/minuscole, trattini e underscore).

c) **Duplicati**: dopo a) e b), si tiene una sola riga (la prima) per ogni coppia data + negozio.

d) **Vendite negative**: si calcola la somma dei tre reparti e, sulle righe valide, il rapporto tipico (mediana) tra somma reparti e vendite totali. Il segno viene invertito solo se il valore assoluto è coerente con la somma dei reparti (scarto entro il 20% dal rapporto tipico); altrimenti il valore diventa vuoto (NaN).

e) **Vendite ×10**: le vendite oltre 3 volte la mediana del proprio negozio vengono divise per 10 solo se il valore diviso è coerente con la somma dei reparti (stesso criterio del punto d); altrimenti NaN.

f) **Assenze**: se `ore_assenza` > `ore_pianificate`, `ore_assenza` diventa NaN.

g) **Scontrini mancanti**: stimati come vendite totali diviso lo scontrino medio del negozio (mediana di vendite/scontrini sulle righe valide dello stesso negozio), arrotondati all'intero. Colonna booleana `scontrini_stimati`. Se anche le vendite sono NaN, resta NaN.

h) **Scarti mancanti**: per ogni reparto, i valori mancanti sono riempiti con la mediana dello stesso reparto, stesso negozio, stesso mese (anno-mese). Colonna booleana `scarti_stimati` (vera se almeno un reparto della riga è stato stimato).

## Sezione 5 – Verifica della pulizia

Importiamo `pulizia.py`, applichiamo le regole al DataFrame originale e controlliamo che il risultato sia quello atteso.

In [13]:
import importlib

import pulizia
importlib.reload(pulizia)  # ricarica il modulo se è stato modificato

df_pulito, report = pulizia.pulisci_dati(df)
print(pulizia.riepilogo_pulizia(report))
print(f"\nRighe: {len(df)} prima, {len(df_pulito)} dopo la pulizia")

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

Righe: 4375 prima, 4332 dopo la pulizia


### Nessuna data non valida, nessun negozio non valido, nessun duplicato data + negozio

In [14]:
date_non_valide = df_pulito["data"].isna().sum()
negozi_non_validi = (~df_pulito["negozio"].isin(NEGOZI_VALIDI)).sum()
duplicati_rimasti = df_pulito.duplicated(subset=["data", "negozio"]).sum()

print("Date non valide:", date_non_valide)
print("Negozi non validi:", negozi_non_validi)
print("Duplicati data + negozio:", duplicati_rimasti)
print("Negozi presenti:", sorted(df_pulito["negozio"].unique()))

assert date_non_valide == 0 and negozi_non_validi == 0 and duplicati_rimasti == 0
print("\nControlli superati")

Date non valide: 0
Negozi non validi: 0
Duplicati data + negozio: 0
Negozi presenti: ['NEG01', 'NEG02', 'NEG03', 'NEG04', 'NEG05', 'NEG06']

Controlli superati


### Valori corretti e scartati nei punti d, e, f

In [15]:
print(f"Rapporto tipico somma reparti / vendite totali: {report['rapporto_tipico_reparti']}\n")
riepilogo_def = pd.DataFrame(
    {
        "corretti": [report["vendite_negative_corrette"], report["vendite_x10_corrette"], 0],
        "scartati (NaN)": [report["vendite_negative_scartate"], report["vendite_x10_scartate"], report["assenze_scartate"]],
    },
    index=["d) vendite negative", "e) vendite x10", "f) ore di assenza"],
)
print(riepilogo_def.to_string())
print()
print("Scontrini stimati:", report["scontrini_stimati"], "| non stimabili:", report["scontrini_non_stimabili"])
print("Righe con scarti stimati:", report["righe_scarti_stimati"])

Rapporto tipico somma reparti / vendite totali: 0.4294

                     corretti  scartati (NaN)
d) vendite negative         5               0
e) vendite x10              5               0
f) ore di assenza           0               5

Scontrini stimati: 43 | non stimabili: 0
Righe con scarti stimati: 127


### Le anomalie di business devono essere ancora presenti

La pulizia deve correggere gli errori dei dati, **non** cancellare i fenomeni reali che l'agente dovrà scoprire.

In [16]:
d = df_pulito

# 1) NEG05: scarti panetteria / vendite panetteria per negozio
per_negozio = d.groupby("negozio")[["scarti_panetteria", "vendite_panetteria"]].sum()
rapporto_pan = per_negozio["scarti_panetteria"] / per_negozio["vendite_panetteria"]
print("1) Scarti / vendite panetteria per negozio:")
print(rapporto_pan.map("{:.1%}".format).to_string())

1) Scarti / vendite panetteria per negozio:
negozio
NEG01     6.1%
NEG02     6.1%
NEG03     6.1%
NEG04     6.1%
NEG05    12.1%
NEG06     6.2%


In [17]:
# 2) Luglio 2025 (7-20): scarti / vendite ortofrutta rispetto a giugno e agosto 2025
def rapporto_orto(maschera):
    righe = d[maschera]
    return righe["scarti_ortofrutta"].sum() / righe["vendite_ortofrutta"].sum()

periodi = {
    "giugno 2025": (d["data"] >= "2025-06-01") & (d["data"] <= "2025-06-30"),
    "luglio 2025, 7-20": (d["data"] >= "2025-07-07") & (d["data"] <= "2025-07-20"),
    "agosto 2025": (d["data"] >= "2025-08-01") & (d["data"] <= "2025-08-31"),
}
print("2) Scarti / vendite ortofrutta:")
for nome, maschera in periodi.items():
    print(f"   {nome:<20} {rapporto_orto(maschera):.1%}")

2) Scarti / vendite ortofrutta:
   giugno 2025          5.7%
   luglio 2025, 7-20    8.5%
   agosto 2025          5.5%


In [18]:
# 3) Marzo 2026: media consegne in ritardo per negozio
marzo = d[(d["data"] >= "2026-03-01") & (d["data"] <= "2026-03-31")]
print("3) Media consegne in ritardo, marzo 2026:")
print(marzo.groupby("negozio")["consegne_in_ritardo"].mean().round(2).to_string())

3) Media consegne in ritardo, marzo 2026:
negozio
NEG01    0.06
NEG02    0.13
NEG03    1.23
NEG04    1.32
NEG05    0.03
NEG06    0.13


In [19]:
# 4) NEG02, settimana 12-18 gennaio 2026: ore di assenza / ore pianificate
neg02 = d[d["negozio"] == "NEG02"]
settimana = neg02[(neg02["data"] >= "2026-01-12") & (neg02["data"] <= "2026-01-18")]
resto = neg02.drop(settimana.index)
print("4) NEG02, ore di assenza / ore pianificate:")
print(f"   12-18 gennaio 2026: {settimana['ore_assenza'].sum() / settimana['ore_pianificate'].sum():.1%}")
print(f"   resto del periodo:  {resto['ore_assenza'].sum() / resto['ore_pianificate'].sum():.1%}")

4) NEG02, ore di assenza / ore pianificate:
   12-18 gennaio 2026: 15.0%
   resto del periodo:  4.5%


## Sezione 6 – Primo test del pandas agent

Creiamo un primo agente con `create_pandas_dataframe_agent` (langchain-experimental 0.4.2) sul DataFrame **pulito** e gli facciamo 4 domande di test.
Il prompt è volutamente minimo: serve a vedere come si comporta l'agente senza istruzioni sul dominio.

Per ogni domanda c'è una **risposta di riferimento** calcolata con pandas, senza agente, considerando che:
- il negozio dell'utente è `NEGOZIO_UTENTE` (NEG01);
- "oggi" è `DATA_RIFERIMENTO`, quindi "questo mese" e "questa settimana" sono gli ultimi periodi disponibili nei dati;
- "questa settimana" = gli ultimi 7 giorni disponibili, confrontati con i 7 giorni precedenti.

### Attenzione: `allow_dangerous_code=True`

Il pandas agent risponde **scrivendo ed eseguendo codice Python generato dal modello** sulla nostra macchina. `allow_dangerous_code=True` è l'autorizzazione esplicita a farlo.

È un rischio perché il codice non viene controllato prima dell'esecuzione: un modello che sbaglia, o un input malevolo (prompt injection), potrebbe leggere o cancellare file, accedere alla rete o eseguire comandi con i permessi dell'utente.

Nel progetto lo accettiamo **solo** perché l'agente gira in locale, su dati simulati, in un prototipo. In un sistema reale servirebbe un ambiente isolato (sandbox) o strumenti con operazioni predefinite.

In [20]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_openai import ChatOpenAI
from langchain_experimental.agents import create_pandas_dataframe_agent

OPENAI_MODEL = os.getenv("OPENAI_MODEL")
if not OPENAI_MODEL or OPENAI_MODEL == "nome-del-modello":
    raise ValueError("OPENAI_MODEL mancante nel file .env")

DATA_RIFERIMENTO = os.getenv("DATA_RIFERIMENTO")
NEGOZIO_UTENTE = os.getenv("NEGOZIO_UTENTE")

# Limiti alle chiamate OpenAI, per contenere i costi durante i test
MAX_PASSI_PER_DOMANDA = 5      # ogni passo dell'agente è una chiamata al modello
MAX_SECONDI_PER_DOMANDA = 60   # tempo massimo per una singola domanda
MAX_CHIAMATE_TOTALI = 20       # tetto complessivo per tutta la sessione del notebook
MAX_TOKEN_RISPOSTA = 800       # lunghezza massima di ogni risposta del modello


class LimiteChiamate(BaseCallbackHandler):
    """Conta le chiamate al modello e blocca l'esecuzione oltre il tetto complessivo."""

    raise_error = True  # l'eccezione interrompe davvero l'agente

    def __init__(self, massimo):
        self.massimo = massimo
        self.chiamate = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        if self.chiamate >= self.massimo:
            raise RuntimeError(f"Limite di {self.massimo} chiamate OpenAI raggiunto")
        self.chiamate += 1


contatore = LimiteChiamate(MAX_CHIAMATE_TOTALI)

llm = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0,
    max_tokens=MAX_TOKEN_RISPOSTA,
    max_retries=0,   # nessun nuovo tentativo automatico in caso di errore
    timeout=30,
    callbacks=[contatore],
)

agente = create_pandas_dataframe_agent(
    llm,
    df_pulito,
    agent_type="tool-calling",
    prefix=f"Oggi è {DATA_RIFERIMENTO}. Rispondi in italiano.",
    verbose=True,
    allow_dangerous_code=True,
    max_iterations=MAX_PASSI_PER_DOMANDA,
    max_execution_time=MAX_SECONDI_PER_DOMANDA,
)
print("Modello:", OPENAI_MODEL, "| Oggi:", DATA_RIFERIMENTO, "| Negozio utente:", NEGOZIO_UTENTE)

C:\Users\maloa\AppData\Local\Temp\ipykernel_5756\3670467623.py:3: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents import create_pandas_dataframe_agent


Modello: gpt-4o-mini | Oggi: 2026-09-01 | Negozio utente: NEG01


In [21]:
# Periodi di riferimento per le risposte calcolate senza agente
mio = df_pulito[df_pulito["negozio"] == NEGOZIO_UTENTE]
ULTIMA_DATA = df_pulito["data"].max()
MESE_CORRENTE = ULTIMA_DATA.to_period("M")
print("Ultima data nei dati:", ULTIMA_DATA.date(), "| Mese corrente:", MESE_CORRENTE)

Ultima data nei dati: 2026-08-31 | Mese corrente: 2026-08


### Domanda 1

> Di quanto siamo sopra o sotto il budget di vendita di questo mese?

In [22]:
risposta = agente.invoke({"input": 'Di quanto siamo sopra o sotto il budget di vendita di questo mese?'})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate finora: {contatore.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Creiamo un DataFrame con i dati forniti\ndata = {\n    'data': ['2024-09-01'] * 5,\n    'negozio': ['NEG01', 'NEG02', 'NEG03', 'NEG04', 'NEG05'],\n    'vendite_totali': [30539.8, 28211.1, 33321.5, 24228.2, 31108.5],\n    'obiettivo_vendite': [32810, 27760, 37850, 23970, 31540]\n}\n\ndf = pd.DataFrame(data)\n\n# Calcoliamo la somma delle vendite totali e degli obiettivi di vendita\nvendite_totali = df['vendite_totali'].sum()\nobiettivo_vendite = df['obiettivo_vendite'].sum()\n\n# Calcoliamo la differenza rispetto al budget\nsopra_sotto_budget = vendite_totali - obiettivo_vendite\nsopra_sotto_budget"}`


-6520.900000000023

Siamo sotto il budget di vendita di questo mese di 6520,90 euro.

> Finished chain.
RISPOSTA FINALE:
Siamo sotto il budget di vendita di questo mese di 6520,90 euro.
Chiamate OpenAI usate finora: 2/20


#### Risposta di riferimento 1

In [22]:
# Risposta di riferimento: vendite del mese corrente di NEG01 contro l'obiettivo
righe = mio[mio["data"].dt.to_period("M") == MESE_CORRENTE]
vendite = righe["vendite_totali"].sum()
obiettivo = righe["obiettivo_vendite"].sum()
scarto = vendite - obiettivo

print(f"{NEGOZIO_UTENTE}, {MESE_CORRENTE} ({len(righe)} giorni)")
print(f"Vendite:   {vendite:>12,.2f} €")
print(f"Obiettivo: {obiettivo:>12,.2f} €")
print(f"Scarto:    {scarto:>+12,.2f} € ({scarto / obiettivo:+.1%})")

NEG01, 2026-08 (31 giorni)
Vendite:     899,860.46 €
Obiettivo:   927,100.00 €
Scarto:      -27,239.54 € (-2.9%)


### Domanda 2

> Le vendite di questa settimana sono calate: dipende dall'affluenza o dallo scontrino medio?

In [24]:
risposta = agente.invoke({"input": "Le vendite di questa settimana sono calate: dipende dall'affluenza o dallo scontrino medio?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate finora: {contatore.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...


Per determinare se la diminuzione delle vendite di questa settimana è dovuta all'affluenza (numero di scontrini) o allo scontrino medio (vendite totali diviso il numero di scontrini), possiamo calcolare e confrontare i seguenti valori:

1. **Affluenza**: il numero totale di scontrini.
2. **Scontrino medio**: vendite totali diviso il numero di scontrini.

Dovremmo confrontare questi valori con quelli della settimana precedente per vedere se ci sono state variazioni significative. Vuoi che esegua questi calcoli per te? Se sì, hai i dati della settimana precedente?

> Finished chain.
RISPOSTA FINALE:
Per determinare se la diminuzione delle vendite di questa settimana è dovuta all'affluenza (numero di scontrini) o allo scontrino medio (vendite totali diviso il numero di scontrini), possiamo calcolare e confrontare i seguenti valori:

1. **Affluenza**: il numero totale di scontrini.
2. **Scontrino medio**: vendite totali diviso il numero di scontrini.

Dovremmo confrontare questi valori con

#### Risposta di riferimento 2

In [23]:
# Risposta di riferimento: ultimi 7 giorni contro i 7 giorni precedenti, per NEG01
def indicatori(inizio, fine):
    righe = mio[(mio["data"] >= inizio) & (mio["data"] <= fine)]
    vendite = righe["vendite_totali"].sum()
    scontrini = righe["numero_scontrini"].sum()
    return pd.Series({
        "vendite": vendite,
        "scontrini": scontrini,
        "scontrino_medio": vendite / scontrini,
        "giorni_con_scontrini_stimati": righe["scontrini_stimati"].sum(),
    })

settimana = indicatori(ULTIMA_DATA - pd.Timedelta(days=6), ULTIMA_DATA)
precedente = indicatori(ULTIMA_DATA - pd.Timedelta(days=13), ULTIMA_DATA - pd.Timedelta(days=7))
confronto = pd.DataFrame({"settimana precedente": precedente, "questa settimana": settimana})
confronto["variazione %"] = (settimana / precedente - 1) * 100
print(f"Questa settimana: {ULTIMA_DATA - pd.Timedelta(days=6):%Y-%m-%d} → {ULTIMA_DATA:%Y-%m-%d}\n")
print(confronto.round(2).to_string())

var_affluenza = confronto.loc["scontrini", "variazione %"]
var_scontrino = confronto.loc["scontrino_medio", "variazione %"]
principale = "affluenza (numero di scontrini)" if abs(var_affluenza) > abs(var_scontrino) else "scontrino medio"
print(f"\nFattore principale della variazione: {principale}")

Questa settimana: 2026-08-25 → 2026-08-31

                              settimana precedente  questa settimana  variazione %
vendite                                  200382.19         203145.19          1.38
scontrini                                  7916.00           7858.00         -0.73
scontrino_medio                              25.31             25.85          2.13
giorni_con_scontrini_stimati                  0.00              0.00           NaN

Fattore principale della variazione: scontrino medio


### Domanda 3

> Quale reparto ha il rapporto più alto tra scarti e vendite nell'ultimo mese?

In [26]:
risposta = agente.invoke({"input": "Quale reparto ha il rapporto più alto tra scarti e vendite nell'ultimo mese?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate finora: {contatore.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Creiamo un DataFrame con i dati forniti\ndata = {\n    'data': ['2024-09-01', '2024-09-01', '2024-09-01', '2024-09-01', '2024-09-01'],\n    'negozio': ['NEG01', 'NEG02', 'NEG03', 'NEG04', 'NEG05'],\n    'vendite_totali': [30539.8, 28211.1, 33321.5, 24228.2, 31108.5],\n    'scarti_ortofrutta': [181.11, 135.78, 175.79, 108.87, 138.94],\n    'scarti_panetteria': [82.38, 102.68, 84.49, 42.46, 185.1],\n    'scarti_freschi': [187.84, 168.66, 199.37, 136.26, 190.22]\n}\n\ndf = pd.DataFrame(data)\n\n# Calcoliamo gli scarti totali e il rapporto scarti/vendite per ogni reparto\nscarti_totali = df[['scarti_ortofrutta', 'scarti_panetteria', 'scarti_freschi']].sum()\nvendite_totali = df['vendite_totali']\n\n# Calcoliamo il rapporto scarti/vendite per ogni reparto\nrapporti = scarti_totali / vendite_totali.sum()\n\n# Troviamo il reparto con il rapporto più alto\nreparto_max_rapporto = rapporti.idxmax(), rapporti.max()\nreparto_ma

Il reparto con il rapporto più alto tra scarti e vendite nell'ultimo mese è il reparto "freschi", con un rapporto di circa 0,00599.

> Finished chain.
RISPOSTA FINALE:
Il reparto con il rapporto più alto tra scarti e vendite nell'ultimo mese è il reparto "freschi", con un rapporto di circa 0,00599.
Chiamate OpenAI usate finora: 5/20


#### Risposta di riferimento 3

In [24]:
# Risposta di riferimento: rapporto scarti / vendite per reparto, NEG01, ultimo mese
righe = mio[mio["data"].dt.to_period("M") == MESE_CORRENTE]
rapporti = pd.Series({
    reparto: righe[f"scarti_{reparto}"].sum() / righe[f"vendite_{reparto}"].sum()
    for reparto in ["ortofrutta", "panetteria", "freschi"]
}).sort_values(ascending=False)
print(f"{NEGOZIO_UTENTE}, {MESE_CORRENTE}: scarti / vendite per reparto")
print(rapporti.map("{:.2%}".format).to_string())
print(f"\nReparto con il rapporto più alto: {rapporti.index[0]}")

NEG01, 2026-08: scarti / vendite per reparto
panetteria    5.97%
ortofrutta    5.68%
freschi       2.45%

Reparto con il rapporto più alto: panetteria


### Domanda 4

> Come sono i nostri scarti rispetto agli altri negozi dell'area?

In [28]:
risposta = agente.invoke({"input": "Come sono i nostri scarti rispetto agli altri negozi dell'area?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate finora: {contatore.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Creiamo un DataFrame con i dati forniti\ndata = {\n    'data': ['2024-09-01'] * 5,\n    'negozio': ['NEG01', 'NEG02', 'NEG03', 'NEG04', 'NEG05'],\n    'vendite_totali': [30539.8, 28211.1, 33321.5, 24228.2, 31108.5],\n    'obiettivo_vendite': [32810, 27760, 37850, 23970, 31540],\n    'numero_scontrini': [1203, 1081, 1354, 976, 1152],\n    'vendite_ortofrutta': [3468.65, 3006.55, 3806.47, 2682.33, 3447.08],\n    'vendite_panetteria': [1568.46, 1451.86, 1789.55, 1210.58, 1683.87],\n    'vendite_freschi': [8341.45, 7854.3, 9050.12, 6249.84, 8710.85],\n    'scarti_ortofrutta': [181.11, 135.78, 175.79, 108.87, 138.94],\n    'scarti_panetteria': [82.38, 102.68, 84.49, 42.46, 185.1],\n    'scarti_freschi': [187.84, 168.66, 199.37, 136.26, 190.22],\n    'scarti_kg_totali': [126.7, 109.8, 126.2, 76.8, 137.2],\n    'referenze_in_rottura': [36, 46, 40, 31, 49],\n    'consegne_in_ritardo': [0, 0, 1, 0, 0],\n    'ore_pianificate'

Ecco come si presentano gli scarti rispetto alla media degli altri negozi:

| Negozio | Scarti Totali (kg) | Scarti Media (kg) | Differenza dalla Media (kg) |
|---------|---------------------|--------------------|------------------------------|
| NEG01   | 126.7               | 115.34             | +11.36                       |
| NEG02   | 109.8               | 115.34             | -5.54                        |
| NEG03   | 126.2               | 115.34             | +10.86                       |
| NEG04   | 76.8                | 115.34             | -38.54                       |
| NEG05   | 137.2               | 115.34             | +21.86                       |

In sintesi:
- **NEG01** e **NEG03** hanno scarti superiori alla media.
- **NEG02** e **NEG04** hanno scarti inferiori alla media, con **NEG04** che presenta la differenza più significativa.
- **NEG05** ha il valore di scarti più alto rispetto alla media.

> Finished chain.
RISPOSTA FINALE:
Ecco come si presentano gli scart

#### Risposta di riferimento 4

In [25]:
# Risposta di riferimento: scarti / vendite dei tre reparti, NEG01 contro gli altri negozi
def rapporto_scarti(righe):
    scarti = righe[["scarti_ortofrutta", "scarti_panetteria", "scarti_freschi"]].sum()
    vendite = righe[["vendite_ortofrutta", "vendite_panetteria", "vendite_freschi"]].sum()
    return pd.Series(scarti.values / vendite.values, index=["ortofrutta", "panetteria", "freschi"]).pipe(
        lambda s: pd.concat([s, pd.Series({"totale": scarti.sum() / vendite.sum()})])
    )

for nome, dati in [(f"ultimo mese ({MESE_CORRENTE})", df_pulito[df_pulito["data"].dt.to_period("M") == MESE_CORRENTE]),
                   ("tutto il periodo", df_pulito)]:
    tabella = dati.groupby("negozio").apply(rapporto_scarti, include_groups=False)
    tabella.loc["media altri negozi"] = tabella.drop(NEGOZIO_UTENTE).mean()
    print(f"Scarti / vendite, {nome}:")
    print(tabella.map("{:.2%}".format).to_string())
    posizione = tabella.drop("media altri negozi")["totale"].rank().loc[NEGOZIO_UTENTE]
    print(f"{NEGOZIO_UTENTE} è {int(posizione)}° su 6 per rapporto totale (1 = più basso)\n")

Scarti / vendite, ultimo mese (2026-08):
                   ortofrutta panetteria freschi totale
negozio                                                
NEG01                   5.68%      5.97%   2.45%  3.68%
NEG02                   5.46%      5.78%   2.19%  3.44%
NEG03                   5.63%      6.04%   2.18%  3.51%
NEG04                   5.73%      6.47%   2.14%  3.56%
NEG05                   5.99%     12.14%   2.25%  4.36%
NEG06                   5.54%      6.40%   2.41%  3.67%
media altri negozi      5.67%      7.36%   2.23%  3.71%
NEG01 è 5° su 6 per rapporto totale (1 = più basso)

Scarti / vendite, tutto il periodo:
                   ortofrutta panetteria freschi totale
negozio                                                
NEG01                   4.88%      6.11%   2.26%  3.38%
NEG02                   4.88%      6.07%   2.25%  3.37%
NEG03                   4.94%      6.12%   2.24%  3.38%
NEG04                   4.92%      6.13%   2.23%  3.37%
NEG05                   4.89% 

## Sezione 7 – Test con il prompt completo

Rifacciamo le stesse 4 domande della Sezione 6 con il prompt completo, salvato in `prompt_data_agent.txt`.
Il prompt definisce ruolo, dati, periodi di tempo, KPI, comportamento, regole di sicurezza e formato della risposta.

I segnaposto (`{oggi}`, `{negozio}`, …) vengono riempiti con `.format()` **prima** di passare il testo all'agente: il `prefix` arriva quindi già completo, senza parentesi graffe da interpretare.
Le risposte di riferimento sono quelle calcolate nella Sezione 6.

*Nota: il file del prompt è stato poi esteso con la sezione METODO (Sezione 8). L'output della cella sotto mostra il prompt usato per il test di questa sezione, senza METODO.*

In [26]:
from datetime import date, timedelta

MESI = ["gennaio", "febbraio", "marzo", "aprile", "maggio", "giugno",
        "luglio", "agosto", "settembre", "ottobre", "novembre", "dicembre"]

# Date di riferimento a partire da DATA_RIFERIMENTO ("oggi" nello scenario simulato)
oggi = date.fromisoformat(DATA_RIFERIMENTO)
ieri = oggi - timedelta(days=1)
inizio_settimana = ieri - timedelta(days=6)
mese_corrente = f"{MESI[ieri.month - 1]} {ieri.year}"

# Carico il testo del prompt e riempio i segnaposto
testo_prompt = (CARTELLA_PROGETTO / "data_agent" / "prompt_data_agent.txt").read_text(encoding="utf-8")
prompt_completo = testo_prompt.format(
    oggi=oggi.isoformat(),
    ieri=ieri.isoformat(),
    inizio_settimana=inizio_settimana.isoformat(),
    mese_corrente=mese_corrente,
    negozio=NEGOZIO_UTENTE,
    riepilogo_pulizia=pulizia.riepilogo_pulizia(report),
)
print(prompt_completo)

RUOLO
Sei l'assistente di analisi dati dello Store Manager del negozio
NEG01, un supermercato di una catena con 6 punti vendita
(NEG01-NEG06) nella stessa area geografica. Il tuo compito è
rispondere alle sue domande calcolando i numeri sui dati reali.
Rispondi sempre in italiano.

DATI
I dati sono già caricati e puliti nella variabile pandas df.
Usa sempre df: non ricrearlo e non copiare mai i valori dall'anteprima.
Ogni riga di df è un negozio in un giorno: le colonne contengono
vendite, obiettivo, scontrini, vendite e scarti per reparto
(ortofrutta, panetteria, freschi), rotture di stock, consegne in
ritardo, ore di lavoro e assenze, eventi del giorno.
Attenzione: vendite_totali è già il totale del negozio. I tre reparti
ne sono solo una parte: non sommarli per ottenere le vendite totali.

TEMPO
Oggi è 2026-09-01. Ieri è 2026-08-31. "Questa settimana" sono i 7 giorni da
2026-08-25 a 2026-08-31. "Questo mese" è agosto 2026, l'ultimo
mese completo nei dati.

KPI (calcolali sempre così

In [27]:
# Nuovo contatore per questa sezione: stesso tetto di chiamate della Sezione 6
contatore_7 = LimiteChiamate(MAX_CHIAMATE_TOTALI)

llm_7 = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0,
    max_tokens=MAX_TOKEN_RISPOSTA,
    max_retries=0,
    timeout=30,
    callbacks=[contatore_7],
)

agente_completo = create_pandas_dataframe_agent(
    llm_7,
    df_pulito,
    agent_type="tool-calling",
    prefix=prompt_completo,
    verbose=True,
    allow_dangerous_code=True,
    max_iterations=MAX_PASSI_PER_DOMANDA,
    max_execution_time=MAX_SECONDI_PER_DOMANDA,
)
print("Agente creato con il prompt completo")

Agente creato con il prompt completo


### Domanda 1 (prompt completo)

> Di quanto siamo sopra o sotto il budget di vendita di questo mese?

Risposta di riferimento: vedi Sezione 6, domanda 1.

In [28]:
risposta = agente_completo.invoke({'input': 'Di quanto siamo sopra o sotto il budget di vendita di questo mese?'})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_7.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "df[(df['data'] >= '2026-08-01') & (df['data'] <= '2026-08-31') & (df['negozio'] == 'NEG01')].agg({'vendite_totali': 'sum', 'obiettivo_vendite': 'sum'})"}`


vendite_totali       899860.46
obiettivo_vendite    927100.00
dtype: float64


Invoking: `python_repl_ast` with `{'query': '(899860.46 - 927100.00) / 927100.00'}`


-0.029381447524538924

1. Scostamento dal budget di vendita per agosto 2026: -2.94%.

2. Siamo sotto il budget di vendita di circa il 2.94% per il mese di agosto 2026.

3. Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
1. Scostamento dal budget di vendita per agosto 2026: -2.94%.

2. Siamo sotto il budget di vendita di circa il 2.94% per il mese di agosto 2026.

3. Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.
Chiamate OpenAI usate in questa sezione: 3/20


### Domanda 2 (prompt completo)

> Le vendite di questa settimana sono calate: dipende dall'affluenza o dallo scontrino medio?

Risposta di riferimento: vedi Sezione 6, domanda 2.

In [29]:
risposta = agente_completo.invoke({'input': "Le vendite di questa settimana sono calate: dipende dall'affluenza o dallo scontrino medio?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_7.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "df[(df['data'] >= '2026-08-25') & (df['data'] <= '2026-08-31') & (df['negozio'] == 'NEG01')]['vendite_totali'].sum()"}`


203145.19
Invoking: `python_repl_ast` with `{'query': "df[(df['data'] >= '2026-08-25') & (df['data'] <= '2026-08-31') & (df['negozio'] == 'NEG01')]['numero_scontrini'].sum()"}`


7858.0
Invoking: `python_repl_ast` with `{'query': "df[(df['data'] >= '2026-08-25') & (df['data'] <= '2026-08-31') & (df['negozio'] == 'NEG01')]['vendite_totali'].sum() / df[(df['data'] >= '2026-08-25') & (df['data'] <= '2026-08-31') & (df['negozio'] == 'NEG01')]['numero_scontrini'].sum()"}`


25.85202214303894

1. Vendite totali questa settimana (25-31 agosto 2026): 203145,19 €
   Affluenza (numero scontrini): 7858
   Scontrino medio: 25,85 €

2. Le vendite non sono calate, anzi, sono state di 203145,19 €, con un'affluenza di 7858 scontrini e uno scontrino medio di 25,85 €. Potrebbe essere utile analizzare ulteriormente se ci sono stati cambiamenti nei reparti o nelle promozioni.

3. Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
1. Vendite totali questa settimana (25-31 agosto 2026): 203145,19 €
   Affluenza (numero scontrini): 7858
   Scontrino medio: 25,85 €

2. Le vendite non sono calate, anzi, sono state di 203145,19 €, con un'affluenza di 7858 scontrini e uno scontrino medio di 25,85 €. Potrebbe essere utile analizza

### Domanda 3 (prompt completo)

> Quale reparto ha il rapporto più alto tra scarti e vendite nell'ultimo mese?

Risposta di riferimento: vedi Sezione 6, domanda 3.

In [30]:
risposta = agente_completo.invoke({'input': "Quale reparto ha il rapporto più alto tra scarti e vendite nell'ultimo mese?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_7.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Filtriamo i dati per il negozio NEG01 e per il mese di agosto 2026\nmese_di_riferimento = df[(df['negozio'] == 'NEG01') & (df['data'].dt.month == 8) & (df['data'].dt.year == 2026)]\n\n# Calcoliamo i tassi di scarto per ogni reparto\nmese_di_riferimento['tasso_scarto_ortofrutta'] = mese_di_riferimento['scarti_ortofrutta'] / mese_di_riferimento['vendite_ortofrutta']\nmese_di_riferimento['tasso_scarto_panetteria'] = mese_di_riferimento['scarti_panetteria'] / mese_di_riferimento['vendite_panetteria']\nmese_di_riferimento['tasso_scarto_freschi'] = mese_di_riferimento['scarti_freschi'] / mese_di_riferimento['vendite_freschi']\n\n# Troviamo il reparto con il tasso di scarto più alto\nrisultati = mese_di_riferimento[['tasso_scarto_ortofrutta', 'tasso_scarto_panetteria', 'tasso_scarto_freschi']].mean()\nreparto_con_scarto_alto = risultati.idxmax(), risultati.max()\nreparto_con_scarto_alto"}`


('tasso_scarto_panetteria', np.

1. Reparto con il rapporto più alto tra scarti e vendite nell'ultimo mese: Panetteria, con un tasso di scarto del 6,00%.

2. Questo indica che il reparto panetteria ha avuto il tasso di scarto più elevato rispetto agli altri reparti, suggerendo la necessità di monitorare meglio le vendite e la gestione degli scarti in questo settore.

3. Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
1. Reparto con il rapporto più alto tra scarti e vendite nell'ultimo mese: Panetteria, con un tasso di scarto del 6,00%.

2. Questo indica che il reparto panetteria ha avuto il tasso di scarto più elevato rispetto agli altri reparti, suggerendo la necessità di monitorare meglio le vendite e la gestione degli scarti in questo settore.

3

### Domanda 4 (prompt completo)

> Come sono i nostri scarti rispetto agli altri negozi dell'area?

Risposta di riferimento: vedi Sezione 6, domanda 4.

In [31]:
risposta = agente_completo.invoke({'input': "Come sono i nostri scarti rispetto agli altri negozi dell'area?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_7.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "df[df['negozio'] == 'NEG01'][['vendite_ortofrutta', 'scarti_ortofrutta', 'vendite_panetteria', 'scarti_panetteria', 'vendite_freschi', 'scarti_freschi']].sum()"}`


vendite_ortofrutta    2189795.300
scarti_ortofrutta      106926.190
vendite_panetteria     998278.400
scarti_panetteria       60992.390
vendite_freschi       5398459.550
scarti_freschi         121937.085
dtype: float64
Invoking: `python_repl_ast` with `{'query': "df[['vendite_ortofrutta', 'scarti_ortofrutta', 'vendite_panetteria', 'scarti_panetteria', 'vendite_freschi', 'scarti_freschi']].sum()"}`


vendite_ortofrutta    1.248968e+07
scarti_ortofrutta     6.123542e+05
vendite_panetteria    5.677839e+06
scarti_panetteria     4.047648e+05
vendite_freschi       3.063232e+07
scarti_freschi        6.861689e+05
dtype: float64


Invoking: `python_repl_ast` with `{'query': '(106926.190 / 2189795.300)'}`


0.048829308383299574
Invoking: `python_repl_ast` with `{'query': '(60992.390 / 998278.400)'}`


0.06109757558612908
Invoking: `python_repl_ast` with `{'query': '(121937.085 / 5398459.550)'}`


0.022587385136561784

Ecco i tassi di scarto per il negozio NEG01 rispetto agli altri negozi dell'area:

- **Ortofrutta**: 4.88%
- **Panetteria**: 6.11%
- **Freschi**: 2.26%

Rispetto agli altri negozi, il tasso di scarto per il reparto ortofrutta è relativamente basso, mentre il reparto panetteria presenta un tasso di scarto più elevato. Il reparto freschi ha un tasso di scarto contenuto.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
Ecco i tassi di scarto per il negozio NEG01 rispetto agli altri negozi dell'area:

- **Ortofrutta**: 4.88%
- **Panetteria**: 6.11%
- **Freschi**: 2.26%

Rispetto agli altri negozi, il tasso di scarto per il reparto ortofrutta è relativamente basso, mentre il reparto panetteria presenta un tasso di scarto 

## Sezione 8 – Test con la sezione METODO

Nel prompt è stata aggiunta la sezione **METODO**, che spiega come fare i confronti (entrambi i termini e la variazione), quale periodo usare quando la domanda non lo indica (ultimi 12 mesi) e come aggregare i tassi (rapporto dei totali).
Rifacciamo tutte e 4 le domande, anche quelle che nella Sezione 7 erano corrette, per verificare di non averle peggiorate.

In [28]:
# Ricarico il prompt dal file (ora contiene la sezione METODO) e riempio i segnaposto come nella Sezione 7
testo_prompt = (CARTELLA_PROGETTO / "data_agent" / "prompt_data_agent.txt").read_text(encoding="utf-8")
prompt_metodo = testo_prompt.format(
    oggi=oggi.isoformat(),
    ieri=ieri.isoformat(),
    inizio_settimana=inizio_settimana.isoformat(),
    mese_corrente=mese_corrente,
    negozio=NEGOZIO_UTENTE,
    riepilogo_pulizia=pulizia.riepilogo_pulizia(report),
)
inizio_metodo = prompt_metodo.index("METODO")
print(prompt_metodo[inizio_metodo:prompt_metodo.index("SICUREZZA")])

METODO
- Confronti: calcola sempre entrambi i termini del confronto e la
  variazione tra i due, prima di trarre conclusioni.
  - Confronto nel tempo: il periodo richiesto e il periodo precedente
    della stessa durata (es. questa settimana e i 7 giorni prima).
  - Confronto con l'area: il valore di NEG01 e la media degli altri
    negozi, calcolata sugli stessi giorni. Se un altro negozio si
    discosta molto dalla media, segnalalo.
- Periodo di default: se la domanda non indica un periodo, usa gli
  ultimi 12 mesi fino a 2026-08-31 e dichiaralo nella risposta.
- Aggregazione: per i tassi su un periodo usa il rapporto dei totali
  (somma degli scarti diviso somma delle vendite), mai la media dei
  rapporti giornalieri.




In [29]:
# Contatore separato per questa sezione, stesso tetto di chiamate
contatore_8 = LimiteChiamate(MAX_CHIAMATE_TOTALI)

llm_8 = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0,
    max_tokens=MAX_TOKEN_RISPOSTA,
    max_retries=0,
    timeout=30,
    callbacks=[contatore_8],
)

agente_metodo = create_pandas_dataframe_agent(
    llm_8,
    df_pulito,
    agent_type="tool-calling",
    prefix=prompt_metodo,
    verbose=True,
    allow_dangerous_code=True,
    max_iterations=MAX_PASSI_PER_DOMANDA,
    max_execution_time=MAX_SECONDI_PER_DOMANDA,
)
print("Agente creato con il prompt con la sezione METODO | Modello:", OPENAI_MODEL)

Agente creato con il prompt con la sezione METODO


### Domanda 1 (con METODO)

> Di quanto siamo sopra o sotto il budget di vendita di questo mese?

Risposta di riferimento: vedi Sezione 6, domanda 1.

In [30]:
risposta = agente_metodo.invoke({'input': 'Di quanto siamo sopra o sotto il budget di vendita di questo mese?'})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_8.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "df[(df['data'] >= '2026-08-01') & (df['data'] <= '2026-08-31') & (df['negozio'] == 'NEG01')].agg({'vendite_totali': 'sum', 'obiettivo_vendite': 'sum'})"}`


vendite_totali       899860.46
obiettivo_vendite    927100.00
dtype: float64


Invoking: `python_repl_ast` with `{'query': '(899860.46 - 927100.00) / 927100.00'}`


-0.029381447524538924

1. Nel mese di agosto 2026, il negozio NEG01 ha registrato vendite totali di 899860,46 € rispetto a un obiettivo di 927100,00 €. Lo scostamento dal budget è di circa -2,94%.

2. Siamo sotto il budget di vendita di questo mese di circa il 2,94%.

3. Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
1. Nel mese di agosto 2026, il negozio NEG01 ha registrato vendite totali di 899860,46 € rispetto a un obiettivo di 927100,00 €. Lo scostamento dal budget è di circa -2,94%.

2. Siamo sotto il budget di vendita di questo mese di circa il 2,94%.

3. Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0)

### Domanda 2 (con METODO)

> Le vendite di questa settimana sono calate: dipende dall'affluenza o dallo scontrino medio?

Risposta di riferimento: vedi Sezione 6, domanda 2.

In [31]:
risposta = agente_metodo.invoke({'input': "Le vendite di questa settimana sono calate: dipende dall'affluenza o dallo scontrino medio?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_8.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\\n\\n# Filtriamo i dati per NEG01 e per la settimana corrente e la settimana precedente\\nthis_week = df[(df['negozio'] == 'NEG01') & (df['data'] >= '2026-08-25') & (df['data'] <= '2026-08-31')]\\nlast_week = df[(df['negozio'] == 'NEG01') & (df['data'] >= '2026-08-18') & (df['data'] <= '2026-08-24')]\\n\\n# Calcoliamo i KPI per questa settimana\\nvendite_totali_this_week = this_week['vendite_totali'].sum()\\nobiettivo_vendite_this_week = this_week['obiettivo_vendite'].sum()\\nnumero_scontrini_this_week = this_week['numero_scontrini'].sum()\\n\\n# Calcoliamo i KPI per la settimana precedente\\nvendite_totali_last_week = last_week['vendite_totali'].sum()\\nobiettivo_vendite_last_week = last_week['obiettivo_vendite'].sum()\\nnumero_scontrini_last_week = last_week['numero_scontrini'].sum()\\n\\n# Calcoliamo lo scostamento dal budget, l'affluenza e lo scontrino medio\\nscostamento_this_week = (vendite_totali_this_week - obiet


Invoking: `python_repl_ast` with `{'query': "# Filtriamo i dati per NEG01 e per la settimana corrente e la settimana precedente\\\\nthis_week = df[(df['negozio'] == 'NEG01') & (df['data'] >= '2026-08-25') & (df['data'] <= '2026-08-31')]\\\\nlast_week = df[(df['negozio'] == 'NEG01') & (df['data'] >= '2026-08-18') & (df['data'] <= '2026-08-24')]\\\\n\\\\n# Calcoliamo i KPI per questa settimana\\\\nvendite_totali_this_week = this_week['vendite_totali'].sum()\\\\nobiettivo_vendite_this_week = this_week['obiettivo_vendite'].sum()\\\\nnumero_scontrini_this_week = this_week['numero_scontrini'].sum()\\\\n\\\\n# Calcoliamo i KPI per la settimana precedente\\\\nvendite_totali_last_week = last_week['vendite_totali'].sum()\\\\nobiettivo_vendite_last_week = last_week['obiettivo_vendite'].sum()\\\\nnumero_scontrini_last_week = last_week['numero_scontrini'].sum()\\\\n\\\\n# Calcoliamo lo scostamento dal budget, l'affluenza e lo scontrino medio\\\\nscostamento_this_week = (vendite_totali_this_week - 


Invoking: `python_repl_ast` with `{'query': "this_week = df[(df['negozio'] == 'NEG01') & (df['data'] >= '2026-08-25') & (df['data'] <= '2026-08-31')]\\\\nlast_week = df[(df['negozio'] == 'NEG01') & (df['data'] >= '2026-08-18') & (df['data'] <= '2026-08-24')]\\\\n\\\\nvendite_totali_this_week = this_week['vendite_totali'].sum()\\\\nvendite_totali_last_week = last_week['vendite_totali'].sum()\\\\nnumero_scontrini_this_week = this_week['numero_scontrini'].sum()\\\\nnumero_scontrini_last_week = last_week['numero_scontrini'].sum()\\\\n\\\\nscontrino_medio_this_week = vendite_totali_this_week / numero_scontrini_this_week if numero_scontrini_this_week > 0 else 0\\\\nscontrino_medio_last_week = vendite_totali_last_week / numero_scontrini_last_week if numero_scontrini_last_week > 0 else 0\\\\n\\\\n(vendite_totali_this_week, vendite_totali_last_week, numero_scontrini_this_week, numero_scontrini_last_week, scontrino_medio_this_week, scontrino_medio_last_week)"}`


SyntaxError: unexpected charact


Invoking: `python_repl_ast` with `{'query': "this_week = df[(df['negozio'] == 'NEG01') & (df['data'] >= '2026-08-25') & (df['data'] <= '2026-08-31')]\\nlast_week = df[(df['negozio'] == 'NEG01') & (df['data'] >= '2026-08-18') & (df['data'] <= '2026-08-24')]\\n\\nvendite_totali_this_week = this_week['vendite_totali'].sum()\\nvendite_totali_last_week = last_week['vendite_totali'].sum()\\nnumero_scontrini_this_week = this_week['numero_scontrini'].sum()\\nnumero_scontrini_last_week = last_week['numero_scontrini'].sum()\\n\\nscontrino_medio_this_week = vendite_totali_this_week / numero_scontrini_this_week if numero_scontrini_this_week > 0 else 0\\nscontrino_medio_last_week = vendite_totali_last_week / numero_scontrini_last_week if numero_scontrini_last_week > 0 else 0\\n\\n(vendite_totali_this_week, vendite_totali_last_week, numero_scontrini_this_week, numero_scontrini_last_week, scontrino_medio_this_week, scontrino_medio_last_week)"}`


SyntaxError: unexpected character after line continua


Invoking: `python_repl_ast` with `{'query': "this_week = df[(df['negozio'] == 'NEG01') & (df['data'] >= '2026-08-25') & (df['data'] <= '2026-08-31')]\\nlast_week = df[(df['negozio'] == 'NEG01') & (df['data'] >= '2026-08-18') & (df['data'] <= '2026-08-24')]\\n\\nvendite_totali_this_week = this_week['vendite_totali'].sum()\\nvendite_totali_last_week = last_week['vendite_totali'].sum()\\nnumero_scontrini_this_week = this_week['numero_scontrini'].sum()\\nnumero_scontrini_last_week = last_week['numero_scontrini'].sum()\\n\\nscontrino_medio_this_week = vendite_totali_this_week / numero_scontrini_this_week if numero_scontrini_this_week > 0 else 0\\nscontrino_medio_last_week = vendite_totali_last_week / numero_scontrini_last_week if numero_scontrini_last_week > 0 else 0\\n\\n(vendite_totali_this_week, vendite_totali_last_week, numero_scontrini_this_week, numero_scontrini_last_week, scontrino_medio_this_week, scontrino_medio_last_week)"}`


SyntaxError: unexpected character after line continua

### Domanda 3 (con METODO)

> Quale reparto ha il rapporto più alto tra scarti e vendite nell'ultimo mese?

Risposta di riferimento: vedi Sezione 6, domanda 3.

In [32]:
risposta = agente_metodo.invoke({'input': "Quale reparto ha il rapporto più alto tra scarti e vendite nell'ultimo mese?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_8.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Filtriamo i dati per il negozio NEG01 e per l'ultimo mese (agosto 2026)\ndf_neg01 = df[(df['negozio'] == 'NEG01') & (df['data'] >= '2026-08-01') & (df['data'] <= '2026-08-31')]\n\n# Calcoliamo i totali per ogni reparto\nscarti_totali = {\n    'ortofrutta': df_neg01['scarti_ortofrutta'].sum(),\n    'panetteria': df_neg01['scarti_panetteria'].sum(),\n    'freschi': df_neg01['scarti_freschi'].sum()\n}\n\nvendite_totali = {\n    'ortofrutta': df_neg01['vendite_ortofrutta'].sum(),\n    'panetteria': df_neg01['vendite_panetteria'].sum(),\n    'freschi': df_neg01['vendite_freschi'].sum()\n}\n\n# Calcoliamo il rapporto scarti/vendite per ogni reparto\nrapporti = {reparto: scarti_totali[reparto] / vendite_totali[reparto] for reparto in scarti_totali.keys()}\n\n# Troviamo il reparto con il rapporto più alto\nreparto_massimo = max(rapporti, key=rapporti.get)\nrapporto_massimo = rapporti[reparto_massimo]\n(reparto_massimo, rapp

1. Reparto con il rapporto più alto tra scarti e vendite nell'ultimo mese (agosto 2026): **Panetteria** con un rapporto di **5.97%**.

2. Questo indica che nel reparto panetteria, il tasso di scarto è relativamente elevato rispetto alle vendite, suggerendo la necessità di monitorare meglio le giacenze e la freschezza dei prodotti.

3. Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
1. Reparto con il rapporto più alto tra scarti e vendite nell'ultimo mese (agosto 2026): **Panetteria** con un rapporto di **5.97%**.

2. Questo indica che nel reparto panetteria, il tasso di scarto è relativamente elevato rispetto alle vendite, suggerendo la necessità di monitorare meglio le giacenze e la freschezza dei prodotti.

3. Puli

### Domanda 4 (con METODO)

> Come sono i nostri scarti rispetto agli altri negozi dell'area?

Risposta di riferimento: ricalcolata qui sotto sugli ultimi 12 mesi.

In [33]:
risposta = agente_metodo.invoke({'input': "Come sono i nostri scarti rispetto agli altri negozi dell'area?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_8.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "df[(df['negozio'] == 'NEG01') & (df['data'] >= '2025-08-01') & (df['data'] <= '2026-08-31')].agg({'scarti_ortofrutta': 'sum', 'vendite_ortofrutta': 'sum', 'scarti_panetteria': 'sum', 'vendite_panetteria': 'sum', 'scarti_freschi': 'sum', 'vendite_freschi': 'sum'})"}`


scarti_ortofrutta       58640.660
vendite_ortofrutta    1204211.140
scarti_panetteria       33700.455
vendite_panetteria     550515.760
scarti_freschi          68166.830
vendite_freschi       2971003.680
dtype: float64
Invoking: `python_repl_ast` with `{'query': "df[(df['data'] >= '2025-08-01') & (df['data'] <= '2026-08-31') & (df['negozio'] != 'NEG01')].agg({'scarti_ortofrutta': 'sum', 'vendite_ortofrutta': 'sum', 'scarti_panetteria': 'sum', 'vendite_panetteria': 'sum', 'scarti_freschi': 'sum', 'vendite_freschi': 'sum'})"}`


scarti_ortofrutta     2.792021e+05
vendite_ortofrutta    5.683548e+06
scarti_panetteria     1.894173e+05
vendite_panetteria    2.583727e+06
scarti_fresc

Ecco i dati sugli scarti per il negozio NEG01 e la media degli altri negozi dell'area, relativi al periodo dal 2025-08-01 al 2026-08-31:

**NEG01:**
- Scarti ortofrutta: 58,640.66
- Vendite ortofrutta: 1,204,211.14
- Scarti panetteria: 33,700.46
- Vendite panetteria: 550,515.76
- Scarti freschi: 68,166.83
- Vendite freschi: 2,971,003.68

**Media degli altri negozi:**
- Scarti ortofrutta: 279,202.10
- Vendite ortofrutta: 5,683,548.00
- Scarti panetteria: 189,417.30
- Vendite panetteria: 2,583,727.00
- Scarti freschi: 311,857.00
- Vendite freschi: 13,940,830.00

**Interpretazione:**
Il negozio NEG01 ha un tasso di scarto significativamente inferiore rispetto alla media degli altri negozi per tutti i reparti. Questo indica una gestione più efficiente degli scarti rispetto alla media dell'area.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore 

#### Risposta di riferimento 4 (ultimi 12 mesi)

Periodo: dal 2025-09-01 al 2026-08-31 (ultimi 12 mesi fino a ieri). Per ogni negozio e reparto il tasso di scarto è il rapporto dei totali (somma scarti / somma vendite del reparto).
La media degli altri negozi è la media semplice dei tassi dei 5 negozi diversi da NEG01.

In [30]:
inizio_12_mesi = pd.Timestamp("2025-09-01")
fine_12_mesi = pd.Timestamp(ieri)
ultimi_12_mesi = df_pulito[(df_pulito["data"] >= inizio_12_mesi) & (df_pulito["data"] <= fine_12_mesi)]

tassi = ultimi_12_mesi.groupby("negozio").apply(rapporto_scarti, include_groups=False)
altri = tassi.drop(NEGOZIO_UTENTE)

print(f"Tasso di scarto per negozio, {inizio_12_mesi:%Y-%m-%d} → {fine_12_mesi:%Y-%m-%d}:")
print(tassi.map("{:.2%}".format).to_string())

confronto_area = pd.DataFrame({
    NEGOZIO_UTENTE: tassi.loc[NEGOZIO_UTENTE],
    "media altri 5": altri.mean(),
})
confronto_area["differenza (punti %)"] = (confronto_area[NEGOZIO_UTENTE] - confronto_area["media altri 5"]) * 100
print(f"\n{NEGOZIO_UTENTE} contro la media degli altri 5 negozi:")
print(confronto_area.to_string(formatters={
    NEGOZIO_UTENTE: "{:.2%}".format, "media altri 5": "{:.2%}".format, "differenza (punti %)": "{:+.2f}".format,
}))

# Negozi che si discostano molto dalla media degli altri (oltre 1,5 volte, per reparto)
media_tutti = tassi.mean()
fuori_scala = (tassi / media_tutti).stack()
fuori_scala = fuori_scala[fuori_scala > 1.5]
print("\nNegozi/reparti oltre 1,5 volte la media dell'area:")
print(fuori_scala.map("{:.2f}x".format).to_string() if len(fuori_scala) else "nessuno")

Tasso di scarto per negozio, 2025-09-01 → 2026-08-31:
        ortofrutta panetteria freschi totale
negozio                                     
NEG01        4.83%      6.12%   2.29%  3.38%
NEG02        4.84%      6.12%   2.25%  3.37%
NEG03        4.83%      6.12%   2.23%  3.35%
NEG04        4.85%      6.24%   2.21%  3.36%
NEG05        4.89%     12.00%   2.26%  4.06%
NEG06        4.87%      6.16%   2.22%  3.36%

NEG01 contro la media degli altri 5 negozi:
           NEG01 media altri 5 differenza (punti %)
ortofrutta 4.83%         4.86%                -0.02
panetteria 6.12%         7.33%                -1.21
freschi    2.29%         2.24%                +0.05
totale     3.38%         3.50%                -0.11

Negozi/reparti oltre 1,5 volte la media dell'area:
negozio            
NEG05    panetteria    1.68x


## Sezione 9 – Stesso prompt, nuovo modello

Stesso prompt della Sezione 8 (con la sezione METODO), stesse domande e stessi limiti: cambia solo il modello, letto da `OPENAI_MODEL` nel `.env` (ora `gpt-4.1-mini`).
Le Sezioni 6, 7 e 8 sono state eseguite con `gpt-4o-mini` e ne conservano gli output per il confronto.

In [31]:
# Contatore separato per questa sezione, stesso tetto di chiamate
# max_tokens più alto: con 800 il codice scritto dall'agente veniva troncato a metà
# (le domande 1-3 di questa sezione sono state eseguite con 800, la domanda 4 con 2000)
MAX_TOKEN_RISPOSTA_9 = 2000
contatore_9 = LimiteChiamate(MAX_CHIAMATE_TOTALI)

llm_9 = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0,
    max_tokens=MAX_TOKEN_RISPOSTA_9,
    max_retries=0,
    timeout=30,
    callbacks=[contatore_9],
)

agente_9 = create_pandas_dataframe_agent(
    llm_9,
    df_pulito,
    agent_type="tool-calling",
    prefix=prompt_metodo,
    verbose=True,
    allow_dangerous_code=True,
    max_iterations=MAX_PASSI_PER_DOMANDA,
    max_execution_time=MAX_SECONDI_PER_DOMANDA,
)
print("Agente creato | Modello:", OPENAI_MODEL)

Agente creato | Modello: gpt-4.1-mini


### Domanda 1 (nuovo modello)

> Di quanto siamo sopra o sotto il budget di vendita di questo mese?

Risposta di riferimento: Sezione 6, domanda 1.

In [32]:
risposta = agente_9.invoke({'input': 'Di quanto siamo sopra o sotto il budget di vendita di questo mese?'})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_9.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Definisco il periodo di questo mese (agosto 2026)\nstart_date = '2026-08-01'\nend_date = '2026-08-31'\n\n# Filtro i dati per il negozio NEG01 e per il periodo di agosto 2026\ndf_negozio_mese = df[(df['negozio'] == 'NEG01') & (df['data'] >= start_date) & (df['data'] <= end_date)]\n\n# Calcolo vendite totali e obiettivo vendite per il periodo\nvendite_totali_mese = df_negozio_mese['vendite_totali'].sum()\nobiettivo_vendite_mese = df_negozio_mese['obiettivo_vendite'].sum()\n\n# Calcolo lo scostamento dal budget\nscostamento_budget = (vendite_totali_mese - obiettivo_vendite_mese) / obiettivo_vendite_mese\n\nvendite_totali_mese, obiettivo_vendite_mese, scostamento_budget"}`


(np.float64(899860.4600000001), np.float64(927100.0), np.float64(-0.0293814475245388))

Nel mese di agosto 2026, per il negozio NEG01, le vendite totali sono state 899.860,46 euro, mentre l'obiettivo di vendita era 927.100,00 euro. Lo scostamento dal budget è quindi di circa -2,94%, cioè siamo sotto il budget di vendita di quasi il 3%.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
Nel mese di agosto 2026, per il negozio NEG01, le vendite totali sono state 899.860,46 euro, mentre l'obiettivo di vendita era 927.100,00 euro. Lo scostamento dal budget è quindi di circa -2,94%, cioè siamo sotto il budget di vendita di quasi il 3%.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartat

### Domanda 2 (nuovo modello)

> Le vendite di questa settimana sono calate: dipende dall'affluenza o dallo scontrino medio?

Risposta di riferimento: Sezione 6, domanda 2.

In [33]:
risposta = agente_9.invoke({'input': "Le vendite di questa settimana sono calate: dipende dall'affluenza o dallo scontrino medio?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_9.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\nfrom datetime import datetime\n\n# Definire le date per questa settimana e la settimana precedente\nstart_this_week = datetime(2026, 8, 25)\nend_this_week = datetime(2026, 8, 31)\nstart_prev_week = datetime(2026, 8, 18)\nend_prev_week = datetime(2026, 8, 24)\n\n# Filtrare i dati per NEG01 e per le due settimane\ndf_neg01 = df[df['negozio'] == 'NEG01']\nthis_week_data = df_neg01[(df_neg01['data'] >= start_this_week) & (df_neg01['data'] <= end_this_week)]\nprev_week_data = df_neg01[(df_neg01['data'] >= start_prev_week) & (df_neg01['data'] <= end_prev_week)]\n\n# Calcolare vendite totali, affluenza (numero_scontrini) e scontrino medio per entrambe le settimane\nvendite_this_week = this_week_data['vendite_totali'].sum()\nvendite_prev_week = prev_week_data['vendite_totali'].sum()\naffluenza_this_week = this_week_data['numero_scontrini'].sum()\naffluenza_prev_week = prev_week_data['numero_scontrini'].sum()\nscontrino_medio_th

Nella settimana dal 25 al 31 agosto 2026, le vendite totali di NEG01 sono state 203.145 euro, in leggero aumento dell'1,38% rispetto alla settimana precedente (200.382 euro). L'affluenza (numero di scontrini) è diminuita dello 0,73% (7.858 contro 7.916), mentre lo scontrino medio è aumentato del 2,13% (25,85 euro contro 25,31 euro).

Quindi, contrariamente a quanto detto, le vendite non sono calate ma sono leggermente aumentate. L'aumento è dovuto principalmente a un incremento dello scontrino medio, non all'affluenza.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
Nella settimana dal 25 al 31 agosto 2026, le vendite totali di NEG01 sono state 203.145 euro, in leggero aumento dell'1,38% rispetto alla settimana prec

### Domanda 3 (nuovo modello)

> Quale reparto ha il rapporto più alto tra scarti e vendite nell'ultimo mese?

Risposta di riferimento: Sezione 6, domanda 3.

In [34]:
risposta = agente_9.invoke({'input': "Quale reparto ha il rapporto più alto tra scarti e vendite nell'ultimo mese?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_9.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Definisco il periodo ultimo mese completo\nultimo_mese = (df['data'] >= '2026-08-01') & (df['data'] <= '2026-08-31')\n\n# Filtro per negozio NEG01 e ultimo mese\ndf_ultimo_mese = df[(df['negozio'] == 'NEG01') & ultimo_mese]\n\n# Calcolo il tasso di scarto per ogni reparto come rapporto tra somma scarti e somma vendite\nscarto_ortofrutta = df_ultimo_mese['scarti_ortofrutta'].sum() / df_ultimo_mese['vendite_ortofrutta'].sum()\nscarto_panetteria = df_ultimo_mese['scarti_panetteria'].sum() / df_ultimo_mese['vendite_panetteria'].sum()\nscarto_freschi = df_ultimo_mese['scarti_freschi'].sum() / df_ultimo_mese['vendite_freschi'].sum()\n\n# Trovo il reparto con il tasso più alto\nreparto_max_scarto = max(\n    [('ortofrutta', scarto_ortofrutta), ('panetteria', scarto_panetteria), ('freschi', scarto_freschi)],\n    key=lambda x: x[1]\n)\n\nreparto_max_scarto"}`


('panetteria', np.float64(0.05972454955114535))

Nell'ultimo mese di agosto 2026, il reparto con il rapporto più alto tra scarti e vendite nel negozio NEG01 è la panetteria, con un tasso di scarto di circa 5,97%.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
Nell'ultimo mese di agosto 2026, il reparto con il rapporto più alto tra scarti e vendite nel negozio NEG01 è la panetteria, con un tasso di scarto di circa 5,97%.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.
Chiamate OpenAI usate in questa sezione: 6/20


### Domanda 4 (nuovo modello)

> Come sono i nostri scarti rispetto agli altri negozi dell'area?

Risposta di riferimento: Sezione 8 (ultimi 12 mesi).

In [32]:
risposta = agente_9.invoke({'input': "Come sono i nostri scarti rispetto agli altri negozi dell'area?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_9.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Filtriamo i dati per l'ultimo mese completo, agosto 2026\ndf['data'] = pd.to_datetime(df['data'])\nultimo_mese = (df['data'].dt.year == 2026) & (df['data'].dt.month == 8)\ndf_agosto = df[ultimo_mese]\n\n# Calcoliamo il tasso di scarto totale per NEG01 e per gli altri negozi\nscarti_neg01 = df_agosto[df_agosto['negozio'] == 'NEG01'][['scarti_ortofrutta', 'scarti_panetteria', 'scarti_freschi', 'vendite_ortofrutta', 'vendite_panetteria', 'vendite_freschi']].sum()\nscarti_altri = df_agosto[df_agosto['negozio'] != 'NEG01'].groupby('negozio')[['scarti_ortofrutta', 'scarti_panetteria', 'scarti_freschi', 'vendite_ortofrutta', 'vendite_panetteria', 'vendite_freschi']].sum()\n\n# Calcolo tassi di scarto per ogni reparto per NEG01\ntasso_scarti_neg01 = {\n    'ortofrutta': scarti_neg01['scarti_ortofrutta'] / scarti_neg01['vendite_ortofrutta'] if scarti_neg01['vendite_ortofrutta'] > 0 else None,\n    'panetteria': scarti_neg01[

Nel mese di agosto 2026, i tassi di scarto per il negozio NEG01 sono stati:
- Ortofrutta: 5,68%
- Panetteria: 5,97%
- Freschi: 2,45%

Per gli altri negozi dell'area nello stesso periodo, i tassi medi di scarto sono stati:
- Ortofrutta: 5,67%
- Panetteria: 7,32%
- Freschi: 2,25%

Interpretazione: NEG01 ha tassi di scarto simili agli altri negozi per ortofrutta, un tasso di scarto più basso in panetteria e leggermente più alto nei freschi rispetto alla media degli altri negozi.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
Nel mese di agosto 2026, i tassi di scarto per il negozio NEG01 sono stati:
- Ortofrutta: 5,68%
- Panetteria: 5,97%
- Freschi: 2,45%

Per gli altri negozi dell'area nello stesso periodo, i tassi m

## Sezione 10 – Test finale del prototipo

Configurazione finale: modello `gpt-4.1-mini` (da `OPENAI_MODEL` nel `.env`), `max_tokens=2000`, altri limiti invariati.
Il prompt è ricaricato dal file, che ora contiene la sezione METODO aggiornata: confronto con l'area negozio per negozio (con segnalazione delle anomalie oltre il 30%) e periodo di default di 12 mesi quando la domanda non indica un periodo.
Tutte e 4 le domande vengono rifatte in un'unica esecuzione.

In [32]:
# Ricarico il prompt aggiornato e riempio i segnaposto come nelle sezioni precedenti
testo_prompt = (CARTELLA_PROGETTO / "data_agent" / "prompt_data_agent.txt").read_text(encoding="utf-8")
prompt_finale = testo_prompt.format(
    oggi=oggi.isoformat(),
    ieri=ieri.isoformat(),
    inizio_settimana=inizio_settimana.isoformat(),
    mese_corrente=mese_corrente,
    negozio=NEGOZIO_UTENTE,
    riepilogo_pulizia=pulizia.riepilogo_pulizia(report),
)
print(prompt_finale[prompt_finale.index("METODO"):prompt_finale.index("SICUREZZA")])

METODO
- Confronti: calcola sempre entrambi i termini del confronto e la
  variazione tra i due, prima di trarre conclusioni.
  - Confronto nel tempo: il periodo richiesto e il periodo precedente
    della stessa durata (es. questa settimana e i 7 giorni prima).
  - Confronto con l'area: calcola il valore di NEG01, il valore di
    OGNI altro negozio e la media degli altri, sugli stessi giorni.
    Mostra una riga per ogni negozio. Se un negozio si discosta dalla
    media di oltre il 30%, segnalalo come possibile anomalia.
- Periodo di default: le definizioni di "ieri", "questa settimana" e
  "questo mese" valgono solo se la domanda usa quelle parole. Se la
  domanda non indica nessun periodo, usa gli ultimi 12 mesi fino a
  2026-08-31 e dichiaralo nella risposta.
- Aggregazione: per i tassi su un periodo usa il rapporto dei totali
  (somma degli scarti diviso somma delle vendite), mai la media dei
  rapporti giornalieri.




In [33]:
MAX_TOKEN_RISPOSTA_10 = 2000

# Contatore separato per questa sezione, stesso tetto di chiamate
contatore_10 = LimiteChiamate(MAX_CHIAMATE_TOTALI)

llm_10 = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0,
    max_tokens=MAX_TOKEN_RISPOSTA_10,
    max_retries=0,
    timeout=30,
    callbacks=[contatore_10],
)

agente_10 = create_pandas_dataframe_agent(
    llm_10,
    df_pulito,
    agent_type="tool-calling",
    prefix=prompt_finale,
    verbose=True,
    allow_dangerous_code=True,
    max_iterations=MAX_PASSI_PER_DOMANDA,
    max_execution_time=MAX_SECONDI_PER_DOMANDA,
)
print("Agente creato | Modello:", OPENAI_MODEL, "| max_tokens:", MAX_TOKEN_RISPOSTA_10)

Agente creato | Modello: gpt-4.1-mini | max_tokens: 2000


### Domanda 1 (test finale)

> Di quanto siamo sopra o sotto il budget di vendita di questo mese?

Risposta di riferimento: Sezione 6, domanda 1.

In [34]:
risposta = agente_10.invoke({'input': 'Di quanto siamo sopra o sotto il budget di vendita di questo mese?'})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_10.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Definisco il periodo di questo mese (agosto 2026)\nstart_date = '2026-08-01'\nend_date = '2026-08-31'\n\n# Filtro i dati per il negozio NEG01 e per il periodo di agosto 2026\ndf_negozio_mese = df[(df['negozio'] == 'NEG01') & (df['data'] >= start_date) & (df['data'] <= end_date)]\n\n# Calcolo vendite totali e obiettivo vendite per il mese\nvendite_totali_mese = df_negozio_mese['vendite_totali'].sum()\nobiettivo_vendite_mese = df_negozio_mese['obiettivo_vendite'].sum()\n\n# Calcolo lo scostamento dal budget\nscostamento_budget = (vendite_totali_mese - obiettivo_vendite_mese) / obiettivo_vendite_mese\n\nvendite_totali_mese, obiettivo_vendite_mese, scostamento_budget"}`


(np.float64(899860.4600000001), np.float64(927100.0), np.float64(-0.0293814475245388))

Nel mese di agosto 2026, per il negozio NEG01, le vendite totali sono state 899.860,46 euro contro un obiettivo di 927.100 euro. Lo scostamento dal budget è quindi di circa -2,94%, cioè siamo sotto il budget di vendita di quasi il 3%.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
Nel mese di agosto 2026, per il negozio NEG01, le vendite totali sono state 899.860,46 euro contro un obiettivo di 927.100 euro. Lo scostamento dal budget è quindi di circa -2,94%, cioè siamo sotto il budget di vendita di quasi il 3%.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenz

### Domanda 2 (test finale)

> Le vendite di questa settimana sono calate: dipende dall'affluenza o dallo scontrino medio?

Risposta di riferimento: Sezione 6, domanda 2.

In [35]:
risposta = agente_10.invoke({'input': "Le vendite di questa settimana sono calate: dipende dall'affluenza o dallo scontrino medio?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_10.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\nfrom datetime import datetime\n\n# Definisco il periodo di questa settimana e della settimana precedente\nstart_this_week = datetime(2026, 8, 25)\nend_this_week = datetime(2026, 8, 31)\nstart_prev_week = datetime(2026, 8, 18)\nend_prev_week = datetime(2026, 8, 24)\n\n# Filtro i dati per NEG01 e per i due periodi\nthis_week_data = df[(df['negozio'] == 'NEG01') & (df['data'] >= start_this_week) & (df['data'] <= end_this_week)]\nprev_week_data = df[(df['negozio'] == 'NEG01') & (df['data'] >= start_prev_week) & (df['data'] <= end_prev_week)]\n\n# Calcolo vendite totali, affluenza e scontrino medio per i due periodi\nvendite_this_week = this_week_data['vendite_totali'].sum()\nvendite_prev_week = prev_week_data['vendite_totali'].sum()\naffluenza_this_week = this_week_data['numero_scontrini'].sum()\naffluenza_prev_week = prev_week_data['numero_scontrini'].sum()\nscontrino_medio_this_week = vendite_this_week / affluenza_this_we

Nella settimana dal 25 al 31 agosto 2026, le vendite totali di NEG01 sono state 203.145 €, in leggero aumento dell'1,38% rispetto alla settimana precedente (200.382 €). L'affluenza (numero di scontrini) è diminuita dello 0,73% (7.858 contro 7.916), mentre lo scontrino medio è aumentato del 2,13% (25,85 € contro 25,31 €).

Quindi, le vendite non sono calate, ma sono leggermente aumentate. L'aumento è dovuto principalmente all'incremento dello scontrino medio, non all'affluenza.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
Nella settimana dal 25 al 31 agosto 2026, le vendite totali di NEG01 sono state 203.145 €, in leggero aumento dell'1,38% rispetto alla settimana precedente (200.382 €). L'affluenza (numero di sco

### Domanda 3 (test finale)

> Quale reparto ha il rapporto più alto tra scarti e vendite nell'ultimo mese?

Risposta di riferimento: Sezione 6, domanda 3.

In [36]:
risposta = agente_10.invoke({'input': "Quale reparto ha il rapporto più alto tra scarti e vendite nell'ultimo mese?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_10.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Definisco il periodo ultimo mese completo\nultimo_mese = '2026-08'\n\n# Filtro per negozio NEG01 e per l'ultimo mese\ndf_ultimo_mese = df[(df['negozio'] == 'NEG01') & (df['data'].dt.strftime('%Y-%m') == ultimo_mese)]\n\n# Calcolo il tasso di scarto per ogni reparto come rapporto tra somma scarti e somma vendite\nscarto_ortofrutta = df_ultimo_mese['scarti_ortofrutta'].sum() / df_ultimo_mese['vendite_ortofrutta'].sum()\nscarto_panetteria = df_ultimo_mese['scarti_panetteria'].sum() / df_ultimo_mese['vendite_panetteria'].sum()\nscarto_freschi = df_ultimo_mese['scarti_freschi'].sum() / df_ultimo_mese['vendite_freschi'].sum()\n\n# Trovo il reparto con il tasso più alto\nreparto_max_scarto = max(\n    ('ortofrutta', scarto_ortofrutta),\n    ('panetteria', scarto_panetteria),\n    ('freschi', scarto_freschi),\n    key=lambda x: x[1]\n)\n\nreparto_max_scarto"}`


('panetteria', np.float64(0.05972454955114535))

Nell'ultimo mese completo (agosto 2026), il reparto con il rapporto più alto tra scarti e vendite nel negozio NEG01 è la panetteria, con un tasso di scarto di circa 5,97%.

Questo indica che, rispetto alle vendite, la panetteria ha avuto la maggiore quantità di prodotti scartati, quindi potrebbe essere utile indagare le cause per ridurre gli sprechi in questo reparto.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
Nell'ultimo mese completo (agosto 2026), il reparto con il rapporto più alto tra scarti e vendite nel negozio NEG01 è la panetteria, con un tasso di scarto di circa 5,97%.

Questo indica che, rispetto alle vendite, la panetteria ha avuto la maggiore quantità di prodotti scartati, quindi potrebbe essere ut

### Domanda 4 (test finale)

> Come sono i nostri scarti rispetto agli altri negozi dell'area?

Risposta di riferimento: Sezione 8 (ultimi 12 mesi).

In [37]:
risposta = agente_10.invoke({'input': "Come sono i nostri scarti rispetto agli altri negozi dell'area?"})
print("RISPOSTA FINALE:", risposta["output"], sep="\n")
print(f"Chiamate OpenAI usate in questa sezione: {contatore_10.chiamate}/{MAX_CHIAMATE_TOTALI}")



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Filtriamo i dati per l'ultimo mese completo, agosto 2026\ndf['data'] = pd.to_datetime(df['data'])\nultimo_mese = (df['data'].dt.year == 2026) & (df['data'].dt.month == 8)\ndf_agosto = df[ultimo_mese]\n\n# Calcoliamo il tasso di scarto totale per ogni negozio come (scarti_kg_totali / (vendite_ortofrutta + vendite_panetteria + vendite_freschi))\n# sommando su tutto il mese\nscarti_area = df_agosto.groupby('negozio').agg({'scarti_kg_totali': 'sum',\n                                                'vendite_ortofrutta': 'sum',\n                                                'vendite_panetteria': 'sum',\n                                                'vendite_freschi': 'sum'}).reset_index()\n\nscarti_area['tasso_scarto'] = scarti_area['scarti_kg_totali'] / (scarti_area['vendite_ortofrutta'] + scarti_area['vendite_panetteria'] + scarti_area['vendite_freschi'])\n\n# Ordiniamo per negozio\nscarti_area = scarti_area.sort_va

Nel mese di agosto 2026, il tasso di scarto totale (scarti in kg diviso vendite totali dei reparti ortofrutta, panetteria e freschi) per NEG01 è stato circa 1,02%. Gli altri negozi dell'area hanno tassi di scarto che variano da circa 0,98% a 1,28%. In particolare, NEG05 ha un tasso di scarto più alto del 30% rispetto alla media degli altri negozi, segnalando una possibile anomalia.

Quindi, i nostri scarti sono nella media dell'area, né particolarmente bassi né alti, ma c'è un negozio con scarti significativamente più elevati.

Pulizia dati: corrette 129 date, uniformati 132 nomi di negozio, rimossi 43 duplicati, corrette 5 vendite negative (scartate 0), corrette 5 vendite moltiplicate per 10 (scartate 0), scartate 5 ore di assenza impossibili, stimati 43 numeri di scontrini, stimati gli scarti in 127 righe.

> Finished chain.
RISPOSTA FINALE:
Nel mese di agosto 2026, il tasso di scarto totale (scarti in kg diviso vendite totali dei reparti ortofrutta, panetteria e freschi) per NEG01 è